# Chào mừng bạn đến với Colab!

In [ ]:
# ============================================================
# STSB IMPROVED LIGHTWEIGHT PIPELINE
# TFIDF + SIAMESE FEATURES + RIDGE
# ============================================================

import os
import time
import joblib
import psutil
import numpy as np

from pathlib import Path

from datasets import load_dataset

from scipy.sparse import hstack, csr_matrix

from scipy.stats import pearsonr, spearmanr

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.pipeline import FeatureUnion

from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_squared_error
)

from sklearn.metrics.pairwise import (
    cosine_similarity
)

# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = "outputs_stsb"

Path(OUTPUT_DIR).mkdir(
    exist_ok=True
)

# ============================================================
# BENCHMARK
# ============================================================

def benchmark(model, X):

    process = psutil.Process(
        os.getpid()
    )

    ram_before = (
        process.memory_info().rss
        / 1024**3
    )

    t0 = time.perf_counter()

    preds = model.predict(X)

    elapsed = (
        time.perf_counter()
        - t0
    )

    ram_after = (
        process.memory_info().rss
        / 1024**3
    )

    n = X.shape[0]

    return {

        "latency_ms":
            (elapsed / max(n, 1))
            * 1000,

        "throughput":
            n / max(elapsed, 1e-9),

        "ram_usage_gb":
            ram_after,

        "ram_delta_gb":
            ram_after - ram_before,
    }

# ============================================================
# LOAD STSB
# ============================================================

print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

raw = load_dataset(
    "glue",
    "stsb"
)

train_a = list(
    raw["train"]["sentence1"]
)

train_b = list(
    raw["train"]["sentence2"]
)

val_a = list(
    raw["validation"]["sentence1"]
)

val_b = list(
    raw["validation"]["sentence2"]
)

# normalize labels 0-5 -> 0-1
y_train = (
    np.asarray(
        raw["train"]["label"]
    ) / 5.0
)

y_val = (
    np.asarray(
        raw["validation"]["label"]
    ) / 5.0
)

print(f"Train samples: {len(train_a)}")
print(f"Validation samples: {len(val_a)}")

# ============================================================
# TFIDF
# ============================================================

print("=" * 60)
print("BUILDING TFIDF")
print("=" * 60)

word_tfidf = TfidfVectorizer(

    analyzer="word",

    lowercase=True,

    stop_words="english",

    max_features=50000,

    ngram_range=(1, 2),

    min_df=1,

    max_df=0.95,

    sublinear_tf=True,
)

char_tfidf = TfidfVectorizer(

    analyzer="char_wb",

    ngram_range=(3, 5),

    max_features=50000,

    sublinear_tf=True,
)

tfidf = FeatureUnion([

    ("word", word_tfidf),

    ("char", char_tfidf),
])

t0 = time.perf_counter()

# ============================================================
# FIT
# ============================================================

X_train_a = tfidf.fit_transform(
    train_a
)

X_train_b = tfidf.transform(
    train_b
)

X_val_a = tfidf.transform(
    val_a
)

X_val_b = tfidf.transform(
    val_b
)

tfidf_time = (
    time.perf_counter()
    - t0
)

print(f"TFIDF done in {tfidf_time:.2f}s")

# ============================================================
# COSINE FEATURES
# ============================================================

print("=" * 60)
print("COSINE FEATURES")
print("=" * 60)

train_cos = cosine_similarity(
    X_train_a,
    X_train_b
).diagonal().reshape(-1, 1)

val_cos = cosine_similarity(
    X_val_a,
    X_val_b
).diagonal().reshape(-1, 1)

# ============================================================
# SIAMESE FEATURES
# ============================================================

print("=" * 60)
print("BUILDING SIAMESE FEATURES")
print("=" * 60)

X_train = hstack([

    X_train_a,

    X_train_b,

    abs(X_train_a - X_train_b),

    X_train_a.multiply(X_train_b),

    csr_matrix(train_cos),
])

X_val = hstack([

    X_val_a,

    X_val_b,

    abs(X_val_a - X_val_b),

    X_val_a.multiply(X_val_b),

    csr_matrix(val_cos),
])

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

# ============================================================
# RAM AFTER TFIDF
# ============================================================

process = psutil.Process(
    os.getpid()
)

ram_after_tfidf = (
    process.memory_info().rss
    / 1024**3
)

# ============================================================
# MODEL
# ============================================================

print("=" * 60)
print("TRAINING RIDGE")
print("=" * 60)

model = Ridge(
    alpha=1.0
)

t0 = time.perf_counter()

model.fit(
    X_train,
    y_train
)

train_time = (
    time.perf_counter()
    - t0
)

ram_after_train = (
    process.memory_info().rss
    / 1024**3
)

print(f"Train time: {train_time:.2f}s")

# ============================================================
# PREDICT
# ============================================================

train_pred = model.predict(
    X_train
)

val_pred = model.predict(
    X_val
)

# back to 0-5
train_pred = np.clip(
    train_pred * 5.0,
    0,
    5
)

val_pred = np.clip(
    val_pred * 5.0,
    0,
    5
)

y_train_real = y_train * 5.0
y_val_real = y_val * 5.0

# ============================================================
# METRICS
# ============================================================

print("=" * 60)
print("EVALUATION")
print("=" * 60)

mse = mean_squared_error(
    y_val_real,
    val_pred
)

pearson = pearsonr(
    y_val_real,
    val_pred
)[0]

spearman = spearmanr(
    y_val_real,
    val_pred
)[0]

print(f"MSE       : {mse:.4f}")
print(f"Pearson   : {pearson:.4f}")
print(f"Spearman  : {spearman:.4f}")

# ============================================================
# BENCHMARK
# ============================================================

print("=" * 60)
print("BENCHMARK")
print("=" * 60)

bench = benchmark(
    model,
    X_val
)

for k, v in bench.items():

    print(f"{k}: {v}")

# ============================================================
# SAVE MODEL
# ============================================================

print("=" * 60)
print("SAVING MODEL")
print("=" * 60)

save_path = (
    Path(OUTPUT_DIR)
    / "stsb_ridge_tfidf.pkl"
)

joblib.dump(

    {
        "tfidf": tfidf,
        "model": model,
    },

    save_path
)

print(f"Saved to: {save_path}")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary = {

    "pearson":
        float(pearson),

    "spearman":
        float(spearman),

    "mse":
        float(mse),

    "tfidf_time":
        tfidf_time,

    "train_time":
        train_time,

    "latency_ms":
        bench["latency_ms"],

    "throughput":
        bench["throughput"],

    "ram_usage_gb":
        bench["ram_usage_gb"],

    "ram_delta_gb":
        bench["ram_delta_gb"],

    "ram_after_tfidf_gb":
        ram_after_tfidf,

    "ram_after_train_gb":
        ram_after_train,
}

for k, v in summary.items():

    print(f"{k}: {v}")

LOADING DATASET
Train samples: 5749
Validation samples: 1500
BUILDING TFIDF
TFIDF done in 3.94s
COSINE FEATURES
BUILDING SIAMESE FEATURES
X_train shape: (5749, 332897)
X_val shape: (1500, 332897)
TRAINING RIDGE
Train time: 1.84s
EVALUATION
MSE       : 0.9600
Pearson   : 0.7637
Spearman  : 0.7636
BENCHMARK
latency_ms: 0.0016917513333586005
throughput: 591103.420627853
ram_usage_gb: 1.953033447265625
ram_delta_gb: 0.0
SAVING MODEL
Saved to: outputs_stsb/stsb_ridge_tfidf.pkl
FINAL SUMMARY
pearson: 0.7637085850175448
spearman: 0.7635707860993352
mse: 0.9600148824650256
tfidf_time: 3.9431062099999963
train_time: 1.8424119369999516
latency_ms: 0.0016917513333586005
throughput: 591103.420627853
ram_usage_gb: 1.953033447265625
ram_delta_gb: 0.0
ram_after_tfidf_gb: 1.9352226257324219
ram_after_train_gb: 1.9530181884765625


<div class="markdown-google-sans">

<a name="machine-learning-examples"></a>

### Ví dụ điển hình

</div>

- <a href="https://docs.jaxstack.ai/en/latest/JAX_for_LLM_pretraining.html">Huấn luyện một mô hình ngôn ngữ miniGPT bằng JAX AI Stack</a>
- <a href="https://github.com/google/tunix/blob/main/examples/qlora_gemma.ipynb">Tinh chỉnh LoRA/QLoRA cho LLM bằng Tunix</a>
- <a href="https://keras.io/examples/keras_recipes/parameter_efficient_finetuning_of_gemma_with_lora_and_qlora/">Tinh chỉnh Gemma một cách hiệu quả về tham số bằng LoRA và QLoRA</a>
- <a href="https://keras.io/keras_hub/guides/hugging_face_keras_integration/">Tải các điểm kiểm tra của Hugging Face Transformers</a>
- <a href="https://keras.io/guides/int8_quantization_in_keras/">Lượng tử hoá số nguyên 8 bit trong Keras</a>
- <a href="https://keras.io/examples/keras_recipes/float8_training_and_inference_with_transformer/">Huấn luyện và suy luận bằng Float8 với một mô hình Transformer đơn giản</a>
- <a href="https://keras.io/keras_hub/guides/transformer_pretraining/">Huấn luyện trước một Transformer từ đầu bằng KerasHub</a>
- <a href="https://keras.io/examples/vision/mnist_convnet/">Mạng nơron tích chập MNIST đơn giản</a>
- <a href="https://keras.io/examples/vision/image_classification_from_scratch/">Phân loại hình ảnh từ đầu bằng Keras 3</a>
- <a href="https://keras.io/keras_hub/guides/classification_with_keras_hub/">Phân loại hình ảnh bằng KerasHub</a>


In [ ]:
# =========================================================
# IMPROVED PURE SVM FOR STS-B
# TF-IDF + Feature Engineering + LinearSVR
# =========================================================

import numpy as np

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVR

from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import hstack

from scipy.stats import pearsonr, spearmanr

# =========================================================
# LOAD DATASET
# =========================================================

print("\nLoading STS-B dataset...")

dataset = load_dataset(
    "glue",
    "stsb"
)

train_ds = dataset["train"]

val_ds = dataset["validation"]

# =========================================================
# EXTRACT TEXT
# =========================================================

train_s1 = list(train_ds["sentence1"])

train_s2 = list(train_ds["sentence2"])

val_s1 = list(val_ds["sentence1"])

val_s2 = list(val_ds["sentence2"])

y_train = np.array(
    train_ds["label"]
)

y_val = np.array(
    val_ds["label"]
)

# =========================================================
# TF-IDF VECTORIZER
# =========================================================

print("\nBuilding TF-IDF vocabulary...")

all_text = train_s1 + train_s2

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    lowercase=True,
)

vectorizer.fit(all_text)

# =========================================================
# TRANSFORM TEXT
# =========================================================

print("\nTransforming training data...")

X1_train = vectorizer.transform(
    train_s1
)

X2_train = vectorizer.transform(
    train_s2
)

print("\nTransforming validation data...")

X1_val = vectorizer.transform(
    val_s1
)

X2_val = vectorizer.transform(
    val_s2
)

# =========================================================
# FEATURE ENGINEERING
# =========================================================

print("\nBuilding similarity features...")

# Cosine similarity
train_cosine = cosine_similarity(
    X1_train,
    X2_train
).diagonal().reshape(-1, 1)

val_cosine = cosine_similarity(
    X1_val,
    X2_val
).diagonal().reshape(-1, 1)

# Absolute difference
train_abs_diff = abs(
    X1_train - X2_train
)

val_abs_diff = abs(
    X1_val - X2_val
)

# Combine features
X_train = hstack([
    X1_train,
    X2_train,
    train_abs_diff,
    train_cosine,
])

X_val = hstack([
    X1_val,
    X2_val,
    val_abs_diff,
    val_cosine,
])

# =========================================================
# MODEL
# =========================================================

print("\nTraining LinearSVR...")

model = LinearSVR(
    C=1.0,
    epsilon=0.1,
    max_iter=5000,
    verbose=1,
)

model.fit(
    X_train,
    y_train,
)

# =========================================================
# PREDICT
# ========================================================



Loading STS-B dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]


Building TF-IDF vocabulary...

Transforming training data...

Transforming validation data...

Building similarity features...

Training LinearSVR...
[LibLinear]

LinearSVR(epsilon=0.1, max_iter=5000, verbose=1)

In [ ]:
print("\nEvaluating...")

preds = model.predict(
    X_val
)

# =========================================================
# METRICS
# =========================================================

pearson_corr = pearsonr(
    y_val,
    preds
)[0]

spearman_corr = spearmanr(
    y_val,
    preds
)[0]

# =========================================================
# RESULTS
# =========================================================

print("\n" + "=" * 60)

print("STS-B RESULTS")

print("=" * 60)

print(f"Pearson  : {pearson_corr:.4f}")

print(f"Spearman : {spearman_corr:.4f}")

print("=" * 60)


Evaluating...

STS-B RESULTS
Pearson  : 0.6891
Spearman : 0.6891


In [ ]:
# ============================================================
# STSB IMPROVED LIGHTWEIGHT PIPELINE
# TFIDF + SIAMESE FEATURES + RIDGE
# ============================================================

import os
import time
import joblib
import psutil
import numpy as np

from pathlib import Path

from datasets import load_dataset

from scipy.sparse import hstack, csr_matrix

from scipy.stats import pearsonr, spearmanr

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.pipeline import FeatureUnion

from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_squared_error
)

from sklearn.metrics.pairwise import (
    cosine_similarity
)

# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = "outputs_stsb"

Path(OUTPUT_DIR).mkdir(
    exist_ok=True
)

# ============================================================
# BENCHMARK
# ============================================================

def benchmark(model, X):

    process = psutil.Process(
        os.getpid()
    )

    ram_before = (
        process.memory_info().rss
        / 1024**3
    )

    t0 = time.perf_counter()

    preds = model.predict(X)

    elapsed = (
        time.perf_counter()
        - t0
    )

    ram_after = (
        process.memory_info().rss
        / 1024**3
    )

    n = X.shape[0]

    return {

        "latency_ms":
            (elapsed / max(n, 1))
            * 1000,

        "throughput":
            n / max(elapsed, 1e-9),

        "ram_usage_gb":
            ram_after,

        "ram_delta_gb":
            ram_after - ram_before,
    }

# ============================================================
# LOAD STSB
# ============================================================

print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

raw = load_dataset(
    "glue",
    "stsb"
)

train_a = list(
    raw["train"]["sentence1"]
)

train_b = list(
    raw["train"]["sentence2"]
)

val_a = list(
    raw["validation"]["sentence1"]
)

val_b = list(
    raw["validation"]["sentence2"]
)

# normalize labels 0-5 -> 0-1
y_train = (
    np.asarray(
        raw["train"]["label"]
    ) / 5.0
)

y_val = (
    np.asarray(
        raw["validation"]["label"]
    ) / 5.0
)

print(f"Train samples: {len(train_a)}")
print(f"Validation samples: {len(val_a)}")

# ============================================================
# TFIDF
# ============================================================

print("=" * 60)
print("BUILDING TFIDF")
print("=" * 60)

word_tfidf = TfidfVectorizer(

    analyzer="word",

    lowercase=True,

    stop_words="english",

    max_features=50000,

    ngram_range=(1, 2),

    min_df=1,

    max_df=0.95,

    sublinear_tf=True,
)

char_tfidf = TfidfVectorizer(

    analyzer="char_wb",

    ngram_range=(3, 5),

    max_features=50000,

    sublinear_tf=True,
)

tfidf = FeatureUnion([

    ("word", word_tfidf),

    ("char", char_tfidf),
])

t0 = time.perf_counter()

# ============================================================
# FIT
# ============================================================

X_train_a = tfidf.fit_transform(
    train_a
)

X_train_b = tfidf.transform(
    train_b
)

X_val_a = tfidf.transform(
    val_a
)

X_val_b = tfidf.transform(
    val_b
)

tfidf_time = (
    time.perf_counter()
    - t0
)

print(f"TFIDF done in {tfidf_time:.2f}s")

# ============================================================
# COSINE FEATURES
# ============================================================

print("=" * 60)
print("COSINE FEATURES")
print("=" * 60)

train_cos = cosine_similarity(
    X_train_a,
    X_train_b
).diagonal().reshape(-1, 1)

val_cos = cosine_similarity(
    X_val_a,
    X_val_b
).diagonal().reshape(-1, 1)

# ============================================================
# SIAMESE FEATURES
# ============================================================

print("=" * 60)
print("BUILDING SIAMESE FEATURES")
print("=" * 60)

X_train = hstack([

    X_train_a,

    X_train_b,

    abs(X_train_a - X_train_b),

    X_train_a.multiply(X_train_b),

    csr_matrix(train_cos),
])

X_val = hstack([

    X_val_a,

    X_val_b,

    abs(X_val_a - X_val_b),

    X_val_a.multiply(X_val_b),

    csr_matrix(val_cos),
])

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

# ============================================================
# RAM AFTER TFIDF
# ============================================================

process = psutil.Process(
    os.getpid()
)

ram_after_tfidf = (
    process.memory_info().rss
    / 1024**3
)

# ============================================================
# MODEL
# ============================================================

print("=" * 60)
print("TRAINING RIDGE")
print("=" * 60)

model = Ridge(
    alpha=1.0
)

t0 = time.perf_counter()

model.fit(
    X_train,
    y_train
)

train_time = (
    time.perf_counter()
    - t0
)

ram_after_train = (
    process.memory_info().rss
    / 1024**3
)

print(f"Train time: {train_time:.2f}s")

# ============================================================
# PREDICT
# ============================================================

train_pred = model.predict(
    X_train
)

val_pred = model.predict(
    X_val
)

# back to 0-5
train_pred = np.clip(
    train_pred * 5.0,
    0,
    5
)

val_pred = np.clip(
    val_pred * 5.0,
    0,
    5
)

y_train_real = y_train * 5.0
y_val_real = y_val * 5.0

# ============================================================
# METRICS
# ============================================================

print("=" * 60)
print("EVALUATION")
print("=" * 60)

mse = mean_squared_error(
    y_val_real,
    val_pred
)

pearson = pearsonr(
    y_val_real,
    val_pred
)[0]

spearman = spearmanr(
    y_val_real,
    val_pred
)[0]

print(f"MSE       : {mse:.4f}")
print(f"Pearson   : {pearson:.4f}")
print(f"Spearman  : {spearman:.4f}")

# ============================================================
# BENCHMARK
# ============================================================

print("=" * 60)
print("BENCHMARK")
print("=" * 60)

bench = benchmark(
    model,
    X_val
)

for k, v in bench.items():

    print(f"{k}: {v}")

# ============================================================
# SAVE MODEL
# ============================================================

print("=" * 60)
print("SAVING MODEL")
print("=" * 60)

save_path = (
    Path(OUTPUT_DIR)
    / "stsb_ridge_tfidf.pkl"
)

joblib.dump(

    {
        "tfidf": tfidf,
        "model": model,
    },

    save_path
)

print(f"Saved to: {save_path}")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

summary = {

    "pearson":
        float(pearson),

    "spearman":
        float(spearman),

    "mse":
        float(mse),

    "tfidf_time":
        tfidf_time,

    "train_time":
        train_time,

    "latency_ms":
        bench["latency_ms"],

    "throughput":
        bench["throughput"],

    "ram_usage_gb":
        bench["ram_usage_gb"],

    "ram_delta_gb":
        bench["ram_delta_gb"],

    "ram_after_tfidf_gb":
        ram_after_tfidf,

    "ram_after_train_gb":
        ram_after_train,
}

for k, v in summary.items():

    print(f"{k}: {v}")

LOADING DATASET
Train samples: 5749
Validation samples: 1500
BUILDING TFIDF
TFIDF done in 3.51s
COSINE FEATURES
BUILDING SIAMESE FEATURES
X_train shape: (5749, 332897)
X_val shape: (1500, 332897)
TRAINING RIDGE
Train time: 3.09s
EVALUATION
MSE       : 0.9600
Pearson   : 0.7637
Spearman  : 0.7636
BENCHMARK
latency_ms: 0.0044319106666534935
throughput: 225636.31697817895
ram_usage_gb: 0.9756050109863281
ram_delta_gb: 0.0
SAVING MODEL
Saved to: outputs_stsb/stsb_ridge_tfidf.pkl
FINAL SUMMARY
pearson: 0.7637085850175448
spearman: 0.7635707860993352
mse: 0.9600148824650256
tfidf_time: 3.508428804999994
train_time: 3.0870456330000025
latency_ms: 0.0044319106666534935
throughput: 225636.31697817895
ram_usage_gb: 0.9756050109863281
ram_delta_gb: 0.0
ram_after_tfidf_gb: 0.960662841796875
ram_after_train_gb: 0.9756050109863281


In [ ]:
# ============================================================
# IMPROVED BiLSTM STSB
# Siamese BiLSTM + TFIDF Similarity Features
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

!pip install -q datasets==2.21.0 scikit-learn scipy psutil

import os
import re
import csv
import json
import time
import random
import psutil

from pathlib import Path

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity

from scipy.stats import pearsonr, spearmanr


# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = "/content/drive/MyDrive/IMPROVED_BILSTM_STSB"

Path(OUTPUT_DIR).mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)

HP = {
    "seed": 42,

    "batch_size": 32,
    "eval_batch_size": 64,

    "max_epochs": 40,
    "patience": 8,

    "lr": 3e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

    "max_seq_len": 64,
    "vocab_size": 50000,

    "embed_dim": 300,
    "hidden": 256,
    "num_layers": 2,
    "dropout": 0.35,

    "num_workers": 2,
}


# ============================================================
# SEED
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(HP["seed"])


# ============================================================
# TOKENIZER + VOCAB
# ============================================================

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return text.split()


class Vocab:

    def __init__(self):
        self.stoi = {
            "<pad>": 0,
            "<unk>": 1
        }

        self.itos = [
            "<pad>",
            "<unk>"
        ]

    def build(self, corpus, max_size):
        freq = {}

        for text in corpus:
            for tok in tokenize(text):
                freq[tok] = freq.get(tok, 0) + 1

        items = sorted(
            freq.items(),
            key=lambda x: x[1],
            reverse=True
        )

        items = items[:max_size - 2]

        for word, _ in items:
            if word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def encode(self, text):
        ids = [
            self.stoi.get(tok, 1)
            for tok in tokenize(text)
        ]

        if len(ids) == 0:
            ids = [1]

        return ids

    def __len__(self):
        return len(self.itos)


# ============================================================
# EXTRA FEATURES
# ============================================================

def token_overlap_features(a_list, b_list):
    feats = []

    for a, b in zip(a_list, b_list):
        ta = set(tokenize(a))
        tb = set(tokenize(b))

        la = len(ta)
        lb = len(tb)

        inter = len(ta & tb)
        union = len(ta | tb)

        jaccard = inter / max(union, 1)
        overlap_a = inter / max(la, 1)
        overlap_b = inter / max(lb, 1)

        len_a = len(tokenize(a))
        len_b = len(tokenize(b))

        len_diff = abs(len_a - len_b) / max(len_a + len_b, 1)
        len_ratio = min(len_a, len_b) / max(max(len_a, len_b), 1)

        feats.append([
            jaccard,
            overlap_a,
            overlap_b,
            len_diff,
            len_ratio
        ])

    return np.asarray(
        feats,
        dtype=np.float32
    )


def build_tfidf_features(train_a, train_b, val_a, val_b):

    print("=" * 60)
    print("BUILDING TFIDF SIDE FEATURES")
    print("=" * 60)

    word_tfidf = TfidfVectorizer(
        analyzer="word",
        lowercase=True,
        stop_words="english",
        max_features=50000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True,
    )

    char_tfidf = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=50000,
        sublinear_tf=True,
    )

    tfidf = FeatureUnion([
        ("word", word_tfidf),
        ("char", char_tfidf),
    ])

    t0 = time.perf_counter()

    X_train_a = tfidf.fit_transform(train_a)
    X_train_b = tfidf.transform(train_b)

    X_val_a = tfidf.transform(val_a)
    X_val_b = tfidf.transform(val_b)

    train_cos = cosine_similarity(
        X_train_a,
        X_train_b
    ).diagonal().reshape(-1, 1)

    val_cos = cosine_similarity(
        X_val_a,
        X_val_b
    ).diagonal().reshape(-1, 1)

    train_overlap = token_overlap_features(
        train_a,
        train_b
    )

    val_overlap = token_overlap_features(
        val_a,
        val_b
    )

    train_feats = np.hstack([
        train_cos.astype(np.float32),
        train_overlap
    ])

    val_feats = np.hstack([
        val_cos.astype(np.float32),
        val_overlap
    ])

    print("TFIDF feature time:", time.perf_counter() - t0)
    print("Extra feature dim:", train_feats.shape[1])

    return tfidf, train_feats, val_feats


# ============================================================
# DATASET
# ============================================================

class STSBDataset(Dataset):

    def __init__(
        self,
        s1,
        s2,
        labels,
        side_feats,
        vocab,
        max_len
    ):
        self.s1 = s1
        self.s2 = s2
        self.labels = labels
        self.side_feats = side_feats
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids1 = self.vocab.encode(
            self.s1[idx]
        )[:self.max_len]

        ids2 = self.vocab.encode(
            self.s2[idx]
        )[:self.max_len]

        return {
            "ids1": torch.tensor(ids1, dtype=torch.long),
            "ids2": torch.tensor(ids2, dtype=torch.long),

            "side_feats": torch.tensor(
                self.side_feats[idx],
                dtype=torch.float
            ),

            "label": torch.tensor(
                float(self.labels[idx]) / 5.0,
                dtype=torch.float
            )
        }


def collate(batch):
    ids1 = [x["ids1"] for x in batch]
    ids2 = [x["ids2"] for x in batch]

    len1 = torch.tensor([
        len(x)
        for x in ids1
    ])

    len2 = torch.tensor([
        len(x)
        for x in ids2
    ])

    ids1 = pad_sequence(
        ids1,
        batch_first=True,
        padding_value=0
    )

    ids2 = pad_sequence(
        ids2,
        batch_first=True,
        padding_value=0
    )

    side_feats = torch.stack([
        x["side_feats"]
        for x in batch
    ])

    labels = torch.stack([
        x["label"]
        for x in batch
    ])

    return {
        "ids1": ids1,
        "ids2": ids2,
        "len1": len1,
        "len2": len2,
        "side_feats": side_feats,
        "labels": labels,
    }


# ============================================================
# MODEL
# ============================================================

class ImprovedBiLSTMSTS(nn.Module):

    def __init__(
        self,
        vocab_size,
        side_dim,
        embed_dim,
        hidden,
        num_layers,
        dropout
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        enc_dim = hidden * 2

        final_dim = (
            enc_dim * 4
            + 1
            + side_dim
        )

        self.regressor = nn.Sequential(
            nn.Linear(final_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def encode(self, ids, lens):
        emb = self.embedding(ids)

        packed = pack_padded_sequence(
            emb,
            lens.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (h, _) = self.lstm(packed)

        h_fw = h[-2]
        h_bw = h[-1]

        return torch.cat(
            [h_fw, h_bw],
            dim=1
        )

    def forward(
        self,
        ids1,
        len1,
        ids2,
        len2,
        side_feats
    ):
        u = self.encode(ids1, len1)
        v = self.encode(ids2, len2)

        diff = torch.abs(u - v)
        prod = u * v

        cos = nn.functional.cosine_similarity(
            u,
            v,
            dim=1
        ).unsqueeze(1)

        feat = torch.cat(
            [
                u,
                v,
                diff,
                prod,
                cos,
                side_feats
            ],
            dim=1
        )

        return self.regressor(feat).squeeze(1)


# ============================================================
# LOSS
# ============================================================

def pearson_loss(pred, target):
    pred = pred - pred.mean()
    target = target - target.mean()

    corr = torch.sum(pred * target) / (
        torch.sqrt(torch.sum(pred ** 2) + 1e-8)
        * torch.sqrt(torch.sum(target ** 2) + 1e-8)
    )

    return 1.0 - corr


def combined_loss(pred, target):
    mse = nn.functional.mse_loss(
        pred,
        target
    )

    corr = pearson_loss(
        pred,
        target
    )

    return mse + 0.15 * corr


# ============================================================
# TRAIN / EVAL
# ============================================================

def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        ids1 = batch["ids1"].to(DEVICE)
        ids2 = batch["ids2"].to(DEVICE)
        len1 = batch["len1"].to(DEVICE)
        len2 = batch["len2"].to(DEVICE)
        side_feats = batch["side_feats"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()

        preds = model(
            ids1,
            len1,
            ids2,
            len2,
            side_feats
        )

        loss = combined_loss(
            preds,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            HP["grad_clip"]
        )

        optimizer.step()

        bs = labels.size(0)

        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / max(total_n, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        ids1 = batch["ids1"].to(DEVICE)
        ids2 = batch["ids2"].to(DEVICE)
        len1 = batch["len1"].to(DEVICE)
        len2 = batch["len2"].to(DEVICE)
        side_feats = batch["side_feats"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        preds = model(
            ids1,
            len1,
            ids2,
            len2,
            side_feats
        )

        loss = nn.functional.mse_loss(
            preds,
            labels
        )

        bs = labels.size(0)

        total_loss += loss.item() * bs
        total_n += bs

        all_preds.extend(
            preds.cpu().numpy().tolist()
        )

        all_labels.extend(
            labels.cpu().numpy().tolist()
        )

    pred = np.asarray(all_preds) * 5.0
    gold = np.asarray(all_labels) * 5.0

    pred = np.clip(
        pred,
        0,
        5
    )

    mse = mean_squared_error(
        gold,
        pred
    )

    mae = mean_absolute_error(
        gold,
        pred
    )

    pearson = pearsonr(
        gold,
        pred
    )[0]

    spearman = spearmanr(
        gold,
        pred
    )[0]

    return {
        "loss": total_loss / max(total_n, 1),
        "mse": float(mse),
        "mae": float(mae),
        "pearson": float(pearson),
        "spearman": float(spearman),
    }


@torch.no_grad()
def benchmark(model, loader):
    model.eval()

    process = psutil.Process(os.getpid())

    ram_before = process.memory_info().rss / 1024**3

    total = 0

    t0 = time.perf_counter()

    for batch in loader:
        ids1 = batch["ids1"].to(DEVICE)
        ids2 = batch["ids2"].to(DEVICE)
        len1 = batch["len1"].to(DEVICE)
        len2 = batch["len2"].to(DEVICE)
        side_feats = batch["side_feats"].to(DEVICE)

        _ = model(
            ids1,
            len1,
            ids2,
            len2,
            side_feats
        )

        total += ids1.size(0)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    ram_after = process.memory_info().rss / 1024**3

    return {
        "latency_ms": float((elapsed / max(total, 1)) * 1000),
        "throughput": float(total / max(elapsed, 1e-9)),
        "ram_usage_gb": float(ram_after),
        "ram_delta_gb": float(ram_after - ram_before),
    }


# ============================================================
# LOAD STSB
# ============================================================

print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

raw = load_dataset(
    "glue",
    "stsb"
)

train_a = list(raw["train"]["sentence1"])
train_b = list(raw["train"]["sentence2"])
val_a = list(raw["validation"]["sentence1"])
val_b = list(raw["validation"]["sentence2"])

y_train = np.asarray(
    raw["train"]["label"],
    dtype=np.float32
)

y_val = np.asarray(
    raw["validation"]["label"],
    dtype=np.float32
)

print("Train samples:", len(train_a))
print("Val samples:", len(val_a))


# ============================================================
# TFIDF SIDE FEATURES
# ============================================================

tfidf, train_side, val_side = build_tfidf_features(
    train_a,
    train_b,
    val_a,
    val_b
)

side_dim = train_side.shape[1]


# ============================================================
# VOCAB
# ============================================================

print("=" * 60)
print("BUILDING VOCAB")
print("=" * 60)

vocab = Vocab()

vocab.build(
    train_a + train_b,
    HP["vocab_size"]
)

print("Vocab size:", len(vocab))


# ============================================================
# DATASET + LOADER
# ============================================================

train_ds = STSBDataset(
    train_a,
    train_b,
    y_train,
    train_side,
    vocab,
    HP["max_seq_len"]
)

val_ds = STSBDataset(
    val_a,
    val_b,
    y_val,
    val_side,
    vocab,
    HP["max_seq_len"]
)

train_loader = DataLoader(
    train_ds,
    batch_size=HP["batch_size"],
    shuffle=True,
    collate_fn=collate,
    num_workers=HP["num_workers"]
)

val_loader = DataLoader(
    val_ds,
    batch_size=HP["eval_batch_size"],
    shuffle=False,
    collate_fn=collate,
    num_workers=HP["num_workers"]
)


# ============================================================
# MODEL
# ============================================================

model = ImprovedBiLSTMSTS(
    vocab_size=len(vocab),
    side_dim=side_dim,
    embed_dim=HP["embed_dim"],
    hidden=HP["hidden"],
    num_layers=HP["num_layers"],
    dropout=HP["dropout"]
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=HP["lr"],
    weight_decay=HP["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


# ============================================================
# TRAINING
# ============================================================

print("=" * 60)
print("TRAINING")
print("=" * 60)

best_score = -1
bad_epochs = 0

best_path = Path(OUTPUT_DIR) / "best_improved_bilstm_stsb.pt"

history = []

t_train = time.perf_counter()

for epoch in range(1, HP["max_epochs"] + 1):

    t0 = time.perf_counter()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    val_metrics = evaluate(
        model,
        val_loader
    )

    score = (
        val_metrics["pearson"]
        + val_metrics["spearman"]
    ) / 2.0

    scheduler.step(score)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
        "score": score,
        "lr": optimizer.param_groups[0]["lr"],
        "time_sec": time.perf_counter() - t0,
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_metrics['loss']:.5f} | "
        f"pearson={val_metrics['pearson']:.4f} | "
        f"spearman={val_metrics['spearman']:.4f} | "
        f"mse={val_metrics['mse']:.4f} | "
        f"score={score:.4f}"
    )

    if score > best_score:
        best_score = score
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "vocab_stoi": vocab.stoi,
                "vocab_itos": vocab.itos,
                "hp": HP,
                "side_dim": side_dim,
                "best_score": best_score,
                "epoch": epoch,
            },
            best_path
        )

        print("Saved best:", best_path)

    else:
        bad_epochs += 1

        if bad_epochs >= HP["patience"]:
            print("Early stopping.")
            break


train_time = time.perf_counter() - t_train


# ============================================================
# FINAL EVAL
# ============================================================

print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

ckpt = torch.load(
    best_path,
    map_location=DEVICE
)

model.load_state_dict(
    ckpt["model_state_dict"]
)

final_metrics = evaluate(
    model,
    val_loader
)

print(json.dumps(
    final_metrics,
    indent=2
))


# ============================================================
# BENCHMARK
# ============================================================

print("=" * 60)
print("BENCHMARK")
print("=" * 60)

bench = benchmark(
    model,
    val_loader
)

print(json.dumps(
    bench,
    indent=2
))


# ============================================================
# SAVE HISTORY + SUMMARY
# ============================================================

history_path = Path(OUTPUT_DIR) / "history.csv"

with open(
    history_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(history[0].keys())
    )

    writer.writeheader()
    writer.writerows(history)


summary = {
    "pearson": final_metrics["pearson"],
    "spearman": final_metrics["spearman"],
    "mse": final_metrics["mse"],
    "mae": final_metrics["mae"],
    "best_score": best_score,
    "train_time": train_time,
    **bench,
    "best_checkpoint": str(best_path),
    "history_path": str(history_path),
}

summary_path = Path(OUTPUT_DIR) / "summary.json"

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )


print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

for k, v in summary.items():
    print(f"{k}: {v}")